<a href="https://colab.research.google.com/github/prof-atritiack/CHECKPOINT_02_SERS_1CC_2SEM/blob/main/Aula_APIs_Energia_Renovavel_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação — APIs de energia renovável e aprendizado de máquina

Consulte duas APIs públicas, organize os dados e resolva **duas tarefas**: classificar a fonte de empreendimentos da ANEEL e estimar a radiação solar em Petrolina (PE). Em **cada tarefa**, treine e compare **três algoritmos diferentes**. Este notebook fornece a consulta às APIs e os arquivos de dados; **você implementará a análise e os modelos**.

**Entrega:** um único link para um repositório público no GitHub com notebook executável, arquivos de dados, README e resultados. As instruções completas estão ao final e no enunciado separado.

## O que é recebido da API?

Uma requisição HTTP `GET` pede dados a uma URL. Os parâmetros identificam o recurso, o período e as variáveis. A resposta é **JSON**: um objeto com chaves e valores. Nas duas consultas desta avaliação, o acesso público **não exige login, token ou chave de API**. Se outra API exigir credenciais, não as publique no repositório.

Este código usa somente `json`, `urllib` e `csv`, bibliotecas que vêm com Python. `csv` serve apenas para exportar os dados recebidos; os imports para análise, tabelas, gráficos e ML são parte da sua implementação.

In [ ]:
import json
import urllib.parse
import urllib.request
import csv

A função a seguir monta a URL, abre a conexão, verifica o status HTTP e converte o JSON. Se houver falha na rede ou na API, exibe uma mensagem de erro.

In [ ]:
def consultar_api(endereco, parametros):
    url = endereco + "?" + urllib.parse.urlencode(parametros)
    try:
        with urllib.request.urlopen(url, timeout=100) as resposta:
            return json.load(resposta)
    except Exception as erro:
        raise RuntimeError(f"Não foi possível consultar a API: {erro}") from erro

# Tarefa 1 — Classificação (ANEEL)

**Pergunta:** a partir da potência e da localização de um empreendimento, é possível classificar sua fonte como **Solar, Eólica ou Hidráulica**?

A fonte é uma **categoria**: o problema é de classificação. O conjunto [SIGA da ANEEL](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel) registra empreendimentos em várias fases. Usamos `UFV` (solar fotovoltaica), `EOL` (eólica) e três categorias hidráulicas: `UHE`, `PCH`, `CGH`. Estas três últimas formarão uma única classe, *Hidráulica*. A tarefa trata da **categoria do cadastro**, não da quantidade de energia gerada.

### Atributos originais da resposta

| Campo da API | Significado | Uso na tarefa |
|---|---|---|
| `SigTipoGeracao` | Sigla do tipo de geração (`UFV`, `EOL`, `UHE`, `PCH`, `CGH`) | Origina o alvo `fonte`; **não entra em X** |
| `MdaPotenciaOutorgadaKw` | Potência outorgada do empreendimento, em quilowatts (kW) | Entrada `potencia_kw`; potência nominal autorizada, não energia produzida |
| `NumCoordNEmpreendimento` | Latitude aproximada do empreendimento, em graus decimais | Entrada `latitude` |
| `NumCoordEEmpreendimento` | Longitude aproximada do empreendimento, em graus decimais | Entrada `longitude` |
| `CodCEG`, `NomEmpreendimento` e campos de combustível | Código, nome e descrição associados ao empreendimento e à fonte | **Não usar como entradas**: podem revelar a classe |

A resposta CKAN contém `success` e `result`; dentro de `result`, `records` são as linhas, `fields` descrevem as colunas e `total` indica o total encontrado para o filtro. O CSV final terá **três atributos numéricos e a classe**. Quantidades consultadas têm limite por tipo e não representam a participação de cada fonte na matriz brasileira.

### 1. Consultar a ANEEL

O recurso SIGA é acessado pela API pública CKAN/DataStore. `resource_id` aponta para o conjunto; `filters` seleciona uma sigla; `limit` limita o retorno de cada pedido. Não há etapa de autenticação para esta consulta pública.

In [ ]:
URL_ANEEL = "https://dadosabertos.aneel.gov.br/api/3/action/datastore_search"
ID_RECURSO = "11ec447d-698d-4ab8-977f-b424d5deee6a"
SIGLAS = ["UFV", "EOL", "UHE", "PCH", "CGH"]
print("API ANEEL: consulta pública, sem token")

In [ ]:
registros_aneel = []
for sigla in SIGLAS:
    parametros = {
        "resource_id": ID_RECURSO,
        "filters": json.dumps({"SigTipoGeracao": sigla}),
        "limit": 1200,
    }
    resposta = consultar_api(URL_ANEEL, parametros)
    if not resposta.get("success"):
        raise ValueError(f"Consulta da ANEEL falhou para {sigla}")
    linhas = resposta["result"]["records"]
    print(f"{sigla}: {len(linhas)} linhas recebidas")
    registros_aneel.extend(linhas)
print("Total de linhas recebidas:", len(registros_aneel))

### 2. Gerar o CSV para a tarefa

A API pode fornecer números como texto, inclusive com vírgula decimal. O tratamento abaixo converte esses valores, descarta linhas sem potência ou coordenadas e transforma as siglas em três classes. Cada linha do arquivo final representa **um empreendimento**. Confira a quantidade de exemplos por classe durante sua análise; se necessário, discuta o desequilíbrio entre classes.

In [ ]:
def numero(valor):
    texto = str(valor).strip()
    if "," in texto:
        texto = texto.replace(".", "").replace(",", ".")
    try:
        return float(texto)
    except ValueError:
        return None

fontes = {"UFV": "Solar", "EOL": "Eólica", "UHE": "Hidráulica",
          "PCH": "Hidráulica", "CGH": "Hidráulica"}
linhas_classificacao = []
for registro in registros_aneel:
    potencia = numero(registro.get("MdaPotenciaOutorgadaKw"))
    latitude = numero(registro.get("NumCoordNEmpreendimento"))
    longitude = numero(registro.get("NumCoordEEmpreendimento"))
    fonte = fontes.get(registro.get("SigTipoGeracao"))
    if fonte and all(valor is not None for valor in (potencia, latitude, longitude)):
        linhas_classificacao.append([potencia, latitude, longitude, fonte])
if len(linhas_classificacao) < 90:
    raise ValueError("Poucos registros válidos: confira a resposta da ANEEL")
print("Linhas válidas:", len(linhas_classificacao))
print("Primeiro exemplo:", linhas_classificacao[0])

In [ ]:
with open("aneel_classificacao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["potencia_kw", "latitude", "longitude", "fonte"])
    escritor.writerows(linhas_classificacao)
print("Salvo: aneel_classificacao_orange.csv")

### O que você deve fazer — classificação

1. Carregue o CSV, confira os tipos das colunas, as contagens por classe, valores ausentes e distribuições relevantes. Explique `X` (três entradas) e `y` (`fonte`).
2. Faça uma divisão **estratificada** de treino/teste, por exemplo 80%/20%, com semente fixa. Ajustes de escala aprendidos dos dados devem ser feitos **somente com o treino**.
3. Escolha, justifique e treine **três classificadores diferentes** usando a mesma divisão. Eles **não estão importados nem implementados** neste notebook.
4. Compare Accuracy, Precision, Recall e F1. Declare o tipo de média usado nas métricas por classe (por exemplo, `macro`) e apresente a matriz de confusão.
5. Explique qual modelo escolheria, quais classes ele mais confunde e por que potência/localização podem não bastar para uma aplicação real. Não use `SigTipoGeracao`, nomes, CEG ou descrições da fonte como entradas.

**O que registrar:** tabela comparativa dos três algoritmos, gráfico(s) ou matriz de confusão e interpretação escrita.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Para classificação: KNN, Regressão Logística, Random Forest
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Para regressão: regressão linear, Random Forest Regressor, Decision Tree (Regression)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# Para classificação
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
# Para regressão
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Caso a consulta da API falhar:
# dados = pd.read_csv('https://raw.githubusercontent.com/prof-atritiack/CHECKPOINT_02_SERS_1CC_2SEM/refs/heads/main/aneel_classificacao_orange.csv')
# dados.head(20)

## Resolução — Tarefa 1 (classificação)

### 1.1 Carregamento e verificação dos dados

O CSV é lido do arquivo gerado pelas células anteriores (consulta à API). Se o arquivo não existir — por exemplo, se a API da ANEEL estiver fora do ar —, é usada a cópia publicada no repositório da disciplina, que tem as mesmas colunas.

In [ ]:
URL_BASE = "https://raw.githubusercontent.com/prof-atritiack/CHECKPOINT_02_SERS_1CC_2SEM/refs/heads/main/"
import os
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

os.makedirs("figuras", exist_ok=True)
SEMENTE = 42

ARQ_CLASS = "aneel_classificacao_orange.csv"
origem = ARQ_CLASS if os.path.exists(ARQ_CLASS) else URL_BASE + ARQ_CLASS
dados = pd.read_csv(origem, encoding="utf-8-sig")
print("Origem dos dados:", origem)
print("Formato:", dados.shape)
dados.info()
dados.head()

In [ ]:
print("Valores ausentes por coluna:")
print(dados.isna().sum(), "\n")

contagem = dados["fonte"].value_counts()
print("Exemplos por classe:")
print(pd.DataFrame({"quantidade": contagem, "proporcao_%": (contagem / len(dados) * 100).round(1)}), "\n")

print("Resumo da potência (kW) por classe:")
dados.groupby("fonte")["potencia_kw"].describe().round(1)

In [ ]:
cores = {"Solar": "tab:orange", "Eólica": "tab:blue", "Hidráulica": "tab:green"}
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

for fonte, grupo in dados.groupby("fonte"):
    eixos[0].hist(np.log10(grupo["potencia_kw"].clip(lower=0.1)), bins=40, alpha=0.55,
                  label=fonte, color=cores[fonte])
eixos[0].set_xlabel("log10(potência outorgada em kW)")
eixos[0].set_ylabel("Quantidade de empreendimentos")
eixos[0].set_title("Distribuição da potência por fonte")
eixos[0].legend()

for fonte, grupo in dados.groupby("fonte"):
    eixos[1].scatter(grupo["longitude"], grupo["latitude"], s=6, alpha=0.5,
                     label=fonte, color=cores[fonte])
eixos[1].set_xlabel("Longitude")
eixos[1].set_ylabel("Latitude")
eixos[1].set_title("Localização dos empreendimentos")
eixos[1].legend(markerscale=3)
plt.tight_layout()
plt.savefig("figuras/aneel_exploracao.png", dpi=120)
plt.show()

**Leitura da exploração.** As três colunas de entrada são numéricas (`float`) e a classe `fonte` é texto. A potência é muito assimétrica (de poucos kW a milhares de MW), por isso o histograma usa escala logarítmica. As classes não são equilibradas: a quantidade de cada sigla foi limitada na consulta (1.200 registros por sigla) e as três siglas hidráulicas (UHE, PCH, CGH) somam-se numa só classe, o que torna *Hidráulica* a maior classe. Essa proporção **não** representa a matriz elétrica brasileira — é um efeito da forma como a amostra foi montada. Por isso, além da Accuracy, olhamos métricas com média `macro`, que dão o mesmo peso às três classes.

No mapa já se vê um padrão: eólicas concentradas no litoral do Nordeste e no Sul, hidráulicas espalhadas pelo Sul/Sudeste/Centro-Oeste e solares espalhadas por quase todo o país.

### 1.2 Definição de X e y e divisão treino/teste

- **X (3 entradas):** `potencia_kw`, `latitude`, `longitude`. A potência entra como `log10(potência)`; essa transformação é uma fórmula fixa (não aprende nada dos dados), então pode ser aplicada antes da divisão sem vazamento.
- **y (alvo):** `fonte` — *Solar*, *Eólica* ou *Hidráulica*.
- Nenhuma coluna que revela a classe (sigla, nome, CEG, combustível) foi usada.

A divisão é **estratificada** (80% treino / 20% teste) com semente fixa, preservando a proporção das classes nos dois conjuntos. A padronização (`StandardScaler`) fica dentro de um `Pipeline`, portanto média e desvio são aprendidos **somente do treino**.

In [ ]:
X_class = pd.DataFrame({
    "log_potencia_kw": np.log10(dados["potencia_kw"].clip(lower=0.1)),
    "latitude": dados["latitude"],
    "longitude": dados["longitude"],
})
y_class = dados["fonte"]

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_class, y_class, test_size=0.2, stratify=y_class, random_state=SEMENTE)

print("Treino:", X_tr_c.shape, "| Teste:", X_te_c.shape)
print(pd.DataFrame({"treino": y_tr_c.value_counts(normalize=True).round(3),
                    "teste": y_te_c.value_counts(normalize=True).round(3)}))

### 1.3 Três classificadores

| Modelo | Por que foi escolhido |
|---|---|
| **KNN** (k = 15, com padronização) | Classifica pela vizinhança: se usinas próximas no mapa e com potência parecida tendem a ser da mesma fonte, o KNN capta isso diretamente. Precisa de escala padronizada porque usa distâncias. |
| **Regressão Logística** (multiclasse, com padronização) | Modelo linear e interpretável; serve como *baseline*. Só consegue separar as classes com fronteiras lineares, o que deve limitar o desempenho em padrões geográficos irregulares. |
| **Random Forest** (300 árvores) | Conjunto de árvores de decisão que capta relações não lineares e interações (ex.: "potência alta **e** litoral do Nordeste"), sem precisar de escala. |

Os três usam a mesma divisão de treino/teste. As métricas por classe usam **média `macro`**.

In [ ]:
classificadores = {
    "KNN (k=15)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "Regressão Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEMENTE, n_jobs=-1),
}

resultados_class = []
previsoes_class = {}
for nome, modelo in classificadores.items():
    modelo.fit(X_tr_c, y_tr_c)
    pred = modelo.predict(X_te_c)
    previsoes_class[nome] = pred
    resultados_class.append({
        "Modelo": nome,
        "Accuracy": accuracy_score(y_te_c, pred),
        "Precision (macro)": precision_score(y_te_c, pred, average="macro"),
        "Recall (macro)": recall_score(y_te_c, pred, average="macro"),
        "F1 (macro)": f1_score(y_te_c, pred, average="macro"),
    })

tabela_class = pd.DataFrame(resultados_class).set_index("Modelo").round(3)
tabela_class.to_csv("figuras/resultados_classificacao.csv")
tabela_class

In [ ]:
classes = ["Solar", "Eólica", "Hidráulica"]
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.8))
for eixo, (nome, pred) in zip(eixos, previsoes_class.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_te_c, pred, labels=classes),
                           display_labels=classes).plot(ax=eixo, cmap="Blues", colorbar=False)
    eixo.set_title(nome)
    eixo.set_xlabel("Classe prevista")
    eixo.set_ylabel("Classe real")
plt.tight_layout()
plt.savefig("figuras/aneel_matrizes_confusao.png", dpi=120)
plt.show()

In [ ]:
melhor_class = tabela_class["F1 (macro)"].idxmax()
print("Melhor modelo pelo F1 macro:", melhor_class, "\n")
print(classification_report(y_te_c, previsoes_class[melhor_class], digits=3))

rf_c = classificadores["Random Forest"]
importancias_c = pd.Series(rf_c.feature_importances_, index=X_class.columns).sort_values()
importancias_c.plot.barh(figsize=(6, 2.8), title="Importância dos atributos — Random Forest")
plt.tight_layout()
plt.savefig("figuras/aneel_importancias.png", dpi=120)
plt.show()

### 1.4 Conclusão — classificação

*(Valores da execução com os dados consultados em out/2026; a base da ANEEL é atualizada continuamente, então os números podem variar levemente em uma nova execução.)*

| Modelo | Accuracy | Precision (macro) | Recall (macro) | F1 (macro) |
|---|---|---|---|---|
| KNN (k=15) | 0,950 | 0,950 | 0,947 | 0,948 |
| Regressão Logística | 0,839 | 0,857 | 0,834 | 0,834 |
| **Random Forest** | **0,977** | **0,978** | **0,976** | **0,977** |

**Modelo escolhido: Random Forest**, com o maior F1 macro (≈ 0,98) e erros baixos e equilibrados nas três classes. O KNN fica próximo (≈ 0,95), o que confirma que "vizinhos no mapa com potência parecida" é uma boa regra. A Regressão Logística fica bem atrás (≈ 0,83): com fronteiras lineares em latitude/longitude ela não representa regiões irregulares, como a faixa litorânea das eólicas, e confunde muitas solares com eólicas (62 casos no teste).

**Quais classes são mais confundidas.** O erro mais frequente do Random Forest é **Solar prevista como Eólica ou Hidráulica** (recall da Solar ≈ 0,95, o menor). Isso acontece com usinas solares de porte médio/grande instaladas nas mesmas regiões de parques eólicos (interior do Nordeste) ou de hidrelétricas (Sudeste/Sul). Já usinas solares muito pequenas (cerca de 64% das solares desta amostra têm até 5 kW) são reconhecidas facilmente só pela potência, porque quase não existem eólicas ou hidráulicas desse tamanho. A importância dos atributos ficou distribuída entre latitude, longitude e potência — nenhum sozinho resolve o problema.

**Por que potência e localização podem não bastar numa aplicação real.**
- O desempenho alto depende de **como a amostra foi montada**: até 1.200 registros por sigla, com proporções que não são as da matriz real. Em outra seleção (ex.: só usinas grandes, ou outra fase do empreendimento) as fronteiras mudam.
- Há sobreposição física: solares e eólicas de dezenas de MW convivem nas mesmas regiões; uma CGH e uma usina solar pequena podem ter a mesma potência e estar no mesmo município.
- Coordenadas cadastradas são **aproximadas** e podem conter erros de cadastro.
- O modelo aprende padrões geográficos do passado; a expansão de fontes para novas regiões (ex.: solar no Sul, eólica offshore) reduziria o acerto.
- Na prática, a fonte de um empreendimento já está no cadastro — o modelo é útil como exercício ou para **verificar inconsistências**, e informações como relevo, presença de rio, irradiação e regime de ventos tornariam a classificação mais robusta.

# Tarefa 2 — Regressão (Open-Meteo)

**Pergunta:** dadas as condições meteorológicas e a hora local em Petrolina (PE), qual é a radiação solar horizontal média naquela hora? A resposta é um **número**, então o problema é de regressão.

A API histórica do [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) devolve dados horários para coordenadas e período escolhidos. Usamos **1º de abril a 30 de junho de 2025**, em horário local. São dados históricos estimados por modelos/reanálise, não leituras de um sensor particular. **Radiação em W/m² não equivale à energia em kWh nem à geração de um painel.**

### Parâmetros da requisição

| Parâmetro | Valor nesta avaliação | Significado |
|---|---|---|
| `latitude`, `longitude` | `-9.39`, `-40.50` | Coordenadas aproximadas de Petrolina (PE) |
| `start_date`, `end_date` | `2025-04-01`, `2025-06-30` | Período histórico consultado, inclusive as datas extremas |
| `timezone` | `America/Recife` | Faz `time` representar a hora local |
| `hourly` | Lista das cinco variáveis abaixo | Solicita valores horários em JSON |

### Atributos do conjunto de dados

| Campo da API → CSV | Significado e unidade | Uso na tarefa |
|---|---|---|
| `time` → `data_hora` | Data e hora local de cada registro | Ordenar e separar por tempo; não é entrada em X |
| `temperature_2m` → `temperatura_c` | Temperatura do ar a 2 m, em °C | Entrada |
| `relative_humidity_2m` → `umidade_pct` | Umidade relativa a 2 m, em % | Entrada |
| `cloud_cover` → `nuvens_pct` | Cobertura total de nuvens, em % | Entrada |
| `wind_speed_10m` → `vento_kmh` | Velocidade do vento a 10 m, em km/h | Entrada |
| `time` → `hora` | Hora local extraída da data, de 7 a 17 nesta amostra | Entrada |
| `shortwave_radiation` → `radiacao_w_m2` | Radiação solar global horizontal média da hora anterior, em W/m² | Alvo numérico `y`; **não entra em X** |

A resposta inclui `hourly` (listas dos valores, alinhadas por índice), `hourly_units` (unidades) e metadados da localização. Cada linha do CSV final representa **uma hora entre 7h e 17h**; esse recorte reduz o domínio de horas noturnas, mas não remove a importância da hora para a previsão.

### 3. Consultar o histórico meteorológico

A chamada usa GET, sem cadastro ou chave para esta consulta pública de uso educacional. Se `hourly` faltar, verifique datas e parâmetros.

In [ ]:
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"
parametros_meteo = {
    "latitude": -9.39,
    "longitude": -40.50,
    "start_date": "2025-04-01",
    "end_date": "2025-06-30",
    "timezone": "America/Recife",
    "hourly": ",".join(["temperature_2m", "relative_humidity_2m",
                         "cloud_cover", "wind_speed_10m", "shortwave_radiation"]),
}
print("Open-Meteo: consulta pública, sem token")

In [ ]:
resposta_meteo = consultar_api(URL_METEO, parametros_meteo)
if "hourly" not in resposta_meteo:
    raise ValueError("Resposta sem a seção hourly")
horas_api = resposta_meteo["hourly"]
print("Campos recebidos:", list(horas_api))
print("Unidades informadas pela API:", resposta_meteo.get("hourly_units", {}))
print("Quantidade de horários:", len(horas_api["time"]))

### 4. Gerar o CSV horário

As listas de `hourly` são alinhadas: o índice 0 de cada variável se refere ao índice 0 de `time`. Percorremos esses índices, mantemos as horas de 7h a 17h e descartamos linhas com valores ausentes. A data permanece ordenada para facilitar uma avaliação temporal.

In [ ]:
nomes_api = ["temperature_2m", "relative_humidity_2m", "cloud_cover",
             "wind_speed_10m", "shortwave_radiation"]
linhas_regressao = []
for indice, data_hora in enumerate(horas_api["time"]):
    hora = int(data_hora[11:13])
    valores = [horas_api[nome][indice] for nome in nomes_api]
    if 7 <= hora <= 17 and all(valor is not None for valor in valores):
        linhas_regressao.append([data_hora, *valores[:4], hora, valores[4]])
if len(linhas_regressao) < 100:
    raise ValueError("Poucos horários válidos: confira a resposta do Open-Meteo")
print("Horas válidas:", len(linhas_regressao))
print("Primeiro exemplo:", linhas_regressao[0])

In [ ]:
with open("meteo_regressao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["data_hora", "temperatura_c", "umidade_pct",
                       "nuvens_pct", "vento_kmh", "hora", "radiacao_w_m2"])
    escritor.writerows(linhas_regressao)
print("Salvo: meteo_regressao_orange.csv")

### O que você deve fazer — regressão

1. Carregue o CSV, explique as colunas, examine valores ausentes e apresente ao menos uma visualização. Defina `X` (cinco entradas) e `y` (`radiacao_w_m2`).
2. Mantenha as linhas em ordem de `data_hora`: use aproximadamente **80% das primeiras horas para treino** e as 20% finais para teste. Evite embaralhar as horas. Ajuste transformações apenas com o treino.
3. Escolha, justifique e treine **três algoritmos de regressão diferentes** com a mesma divisão. Os imports e a implementação são sua responsabilidade.
4. Compare **MAE** (W/m²), **MSE** ((W/m²)²) e **R²**; faça um gráfico de valores reais × previstos de pelo menos um modelo.
5. Interprete os erros, discuta o peso da hora e explique por que essa estimativa da radiação **não prevê automaticamente a energia produzida** por um sistema fotovoltaico.

**O que registrar:** tabela comparativa dos três algoritmos, gráfico e interpretação escrita. Nenhum valor de `radiacao_w_m2` deve ser usado para criar uma entrada do mesmo registro.

## Resolução — Tarefa 2 (regressão)

### 2.1 Carregamento, colunas e valores ausentes

Mesma lógica da Tarefa 1: usa o CSV gerado pela consulta ao Open-Meteo e, se ele não existir, a cópia do repositório da disciplina.

| Coluna | Significado | Papel |
|---|---|---|
| `data_hora` | Data e hora local (America/Recife) | Ordenação e divisão temporal; **não** entra em X |
| `temperatura_c` | Temperatura do ar a 2 m (°C) | Entrada |
| `umidade_pct` | Umidade relativa a 2 m (%) | Entrada |
| `nuvens_pct` | Cobertura total de nuvens (%) | Entrada |
| `vento_kmh` | Velocidade do vento a 10 m (km/h) | Entrada |
| `hora` | Hora local, de 7 a 17 | Entrada |
| `radiacao_w_m2` | Radiação global horizontal média da hora anterior (W/m²) | Alvo `y` |

In [ ]:
ARQ_REG = "meteo_regressao_orange.csv"
origem = ARQ_REG if os.path.exists(ARQ_REG) else URL_BASE + ARQ_REG
meteo = pd.read_csv(origem, encoding="utf-8-sig", parse_dates=["data_hora"])
meteo = meteo.sort_values("data_hora").reset_index(drop=True)
print("Origem dos dados:", origem)
print("Formato:", meteo.shape)
print("Período:", meteo["data_hora"].min(), "→", meteo["data_hora"].max(), "\n")
print("Valores ausentes por coluna:")
print(meteo.isna().sum(), "\n")
meteo.describe().round(1)

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(17, 4.6))

media_hora = meteo.groupby("hora")["radiacao_w_m2"].agg(["mean", "std"])
eixos[0].errorbar(media_hora.index, media_hora["mean"], yerr=media_hora["std"], marker="o", capsize=3)
eixos[0].set_xlabel("Hora local")
eixos[0].set_ylabel("Radiação (W/m²)")
eixos[0].set_title("Radiação média (± desvio) por hora")

sc = eixos[1].scatter(meteo["nuvens_pct"], meteo["radiacao_w_m2"], c=meteo["hora"], cmap="viridis", s=8)
eixos[1].set_xlabel("Cobertura de nuvens (%)")
eixos[1].set_ylabel("Radiação (W/m²)")
eixos[1].set_title("Radiação × nuvens (cor = hora)")
fig.colorbar(sc, ax=eixos[1], label="hora")

corr = meteo.drop(columns="data_hora").corr()
im = eixos[2].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
eixos[2].set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
eixos[2].set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        eixos[2].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
eixos[2].set_title("Correlação de Pearson")
plt.tight_layout()
plt.savefig("figuras/meteo_exploracao.png", dpi=120)
plt.show()

**Leitura da exploração.** Não há valores ausentes (as linhas incompletas já haviam sido descartadas na geração do CSV). A radiação desenha uma "curva de sino" ao longo do dia, com pico perto do meio-dia: a relação com a hora é forte, mas **não linear** (7h e 17h têm radiação baixa; 12h, alta). Por isso a correlação de Pearson entre `hora` e radiação é pequena, mesmo a hora sendo decisiva. Dentro de uma mesma hora, mais nuvens tendem a reduzir a radiação.

### 2.2 Definição de X e y e divisão temporal

- **X (5 entradas):** `temperatura_c`, `umidade_pct`, `nuvens_pct`, `vento_kmh`, `hora`.
- **y:** `radiacao_w_m2`. Nenhuma entrada é derivada da radiação.

As linhas ficam em ordem cronológica: as **primeiras 80% das horas** são treino e as **últimas 20%** são teste, sem embaralhar. Assim o modelo é avaliado em dias "futuros", como seria num uso real, e evita-se que horas vizinhas (muito parecidas) caiam em treino e teste ao mesmo tempo. A padronização usada pela Regressão Linear é ajustada só com o treino (dentro do `Pipeline`).

In [ ]:
colunas_X = ["temperatura_c", "umidade_pct", "nuvens_pct", "vento_kmh", "hora"]
X_reg = meteo[colunas_X]
y_reg = meteo["radiacao_w_m2"]

corte = int(len(meteo) * 0.8)
X_tr_r, X_te_r = X_reg.iloc[:corte], X_reg.iloc[corte:]
y_tr_r, y_te_r = y_reg.iloc[:corte], y_reg.iloc[corte:]
datas_te = meteo["data_hora"].iloc[corte:]

print("Treino:", len(X_tr_r), "horas |", meteo["data_hora"].iloc[0], "→", meteo["data_hora"].iloc[corte - 1])
print("Teste: ", len(X_te_r), "horas |", meteo["data_hora"].iloc[corte], "→", meteo["data_hora"].iloc[-1])

### 2.3 Três regressores

| Modelo | Por que foi escolhido |
|---|---|
| **Regressão Linear** (com padronização) | *Baseline* simples e interpretável. Como trata a hora de forma linear, não consegue representar a curva de sino — serve justamente para mostrar esse limite. |
| **Árvore de Decisão** (profundidade máx. 6) | Divide os dados em faixas (ex.: "hora entre 10 e 14 **e** nuvens < 60%"), captando a não linearidade da hora. A profundidade limitada reduz o sobreajuste. |
| **Random Forest** (300 árvores, mín. 5 amostras por folha) | Média de muitas árvores: mantém a capacidade não linear e reduz a variância de uma árvore isolada. |

Mesma divisão temporal para os três. Métricas: **MAE** (W/m²), **MSE** ((W/m²)²) e **R²**.

In [ ]:
regressores = {
    "Regressão Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Árvore de Decisão": DecisionTreeRegressor(max_depth=6, random_state=SEMENTE),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                           random_state=SEMENTE, n_jobs=-1),
}

resultados_reg = []
previsoes_reg = {}
for nome, modelo in regressores.items():
    modelo.fit(X_tr_r, y_tr_r)
    pred = modelo.predict(X_te_r)
    previsoes_reg[nome] = pred
    resultados_reg.append({
        "Modelo": nome,
        "MAE (W/m²)": mean_absolute_error(y_te_r, pred),
        "MSE ((W/m²)²)": mean_squared_error(y_te_r, pred),
        "RMSE (W/m²)": np.sqrt(mean_squared_error(y_te_r, pred)),
        "R²": r2_score(y_te_r, pred),
    })

tabela_reg = pd.DataFrame(resultados_reg).set_index("Modelo").round(3)
tabela_reg.to_csv("figuras/resultados_regressao.csv")
tabela_reg

In [ ]:
melhor_reg = tabela_reg["R²"].idxmax()
print("Melhor modelo pelo R²:", melhor_reg)

fig, eixos = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 2]})
limite = max(y_te_r.max(), max(p.max() for p in previsoes_reg.values())) * 1.05
for nome, pred in previsoes_reg.items():
    eixos[0].scatter(y_te_r, pred, s=10, alpha=0.6, label=nome)
eixos[0].plot([0, limite], [0, limite], "k--", label="previsão perfeita")
eixos[0].set_xlabel("Radiação real (W/m²)")
eixos[0].set_ylabel("Radiação prevista (W/m²)")
eixos[0].set_title("Real × previsto (conjunto de teste)")
eixos[0].legend(fontsize=8)

n = 11 * 7  # ~7 dias de horas diurnas
eixos[1].plot(range(n), y_te_r.values[:n], "k-o", ms=3, label="Real")
eixos[1].plot(range(n), previsoes_reg[melhor_reg][:n], "-", label=f"Previsto — {melhor_reg}")
eixos[1].plot(range(n), previsoes_reg["Regressão Linear"][:n], "--", alpha=0.7, label="Previsto — Regressão Linear")
inicio_dia = [i for i, d in enumerate(datas_te.iloc[:n]) if d.hour == 7]
eixos[1].set_xticks(inicio_dia, datas_te.iloc[inicio_dia].dt.strftime("%d/%m"))
eixos[1].set_xlabel("Dia (horas de 7h a 17h)")
eixos[1].set_ylabel("Radiação (W/m²)")
eixos[1].set_title("Primeiros 7 dias do período de teste")
eixos[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("figuras/meteo_real_vs_previsto.png", dpi=120)
plt.show()

In [ ]:
rf_r = regressores["Random Forest"]
importancias_r = pd.Series(rf_r.feature_importances_, index=colunas_X).sort_values()
importancias_r.plot.barh(figsize=(6, 3), title="Importância dos atributos — Random Forest")
plt.tight_layout()
plt.savefig("figuras/meteo_importancias.png", dpi=120)
plt.show()

erro_abs = pd.Series(np.abs(y_te_r.values - previsoes_reg[melhor_reg]), index=X_te_r["hora"].values)
print(f"MAE do {melhor_reg} por hora do dia (W/m²):")
print(erro_abs.groupby(level=0).mean().round(1).to_string())

### 2.4 Conclusão — regressão

*(Dados históricos do Open-Meteo, 01/04 a 30/06/2025; treino até 12/06 às 14h e teste de 12/06 às 15h até 30/06.)*

| Modelo | MAE (W/m²) | MSE ((W/m²)²) | RMSE (W/m²) | R² |
|---|---|---|---|---|
| Regressão Linear | 145,2 | 30.034 | 173,3 | 0,360 |
| Árvore de Decisão | 90,9 | 15.127 | 123,0 | 0,678 |
| **Random Forest** | **72,4** | **8.411** | **91,7** | **0,821** |

**Interpretação dos erros.** O Random Forest explica cerca de 82% da variação da radiação no período de teste e erra, em média, ~72 W/m² — para uma radiação média de ~470 W/m², um erro relativo da ordem de 15%. O MSE penaliza mais os erros grandes; a diferença entre MAE e RMSE indica alguns erros bem maiores que a média, típicos de horas com nuvens passageiras (o valor horário de nuvens não diz se a nuvem estava na frente do Sol). O erro por hora mostra que o modelo acerta melhor nos extremos do dia (7h–8h, radiação baixa) e erra mais entre 10h e 15h, quando a radiação é alta e a nebulosidade faz mais diferença em W/m².

**Peso da hora.** A hora é o atributo mais importante no Random Forest (~50% da importância), mesmo tendo correlação linear de apenas 0,12 com a radiação. Isso acontece porque a relação é em forma de sino (sobe até o meio-dia e desce depois). A Regressão Linear só consegue representar "quanto mais tarde, mais (ou menos) radiação", por isso tem o pior resultado (R² ≈ 0,36) e chega a prever **radiação negativa** em algumas horas, algo fisicamente impossível; árvores conseguem criar faixas de horário. Temperatura e umidade vêm em seguida: elas variam junto com o aquecimento do dia e acabam funcionando, em parte, como "indicadores indiretos" do horário e do céu limpo. Uma melhoria simples seria transformar a hora em atributos cíclicos (seno/cosseno) ou usar a elevação solar.

**Por que essa estimativa não prevê automaticamente a energia produzida por um sistema fotovoltaico.**
- O alvo é **radiação global horizontal média em W/m²** (potência por área), não **energia em kWh**. Energia exige integrar ao longo do tempo e multiplicar pela área e pela eficiência do sistema.
- Painéis normalmente são **inclinados e orientados**; a irradiância no plano do módulo difere da horizontal (componentes direta e difusa, ângulo de incidência).
- A geração depende da **temperatura das células** (eficiência cai com o calor), do inversor, de sombreamento, sujeira, perdas em cabos e degradação, além de limitações da rede (*curtailment*).
- Os dados são **estimativas de modelo/reanálise** para um ponto aproximado de Petrolina, não medições de um sensor no local da usina.
- O teste cobre apenas ~3 semanas de junho (período seco); o desempenho pode ser diferente na estação chuvosa.

# Considerações finais

| Tarefa | Melhor modelo | Métrica principal |
|---|---|---|
| Classificação da fonte (ANEEL) | Random Forest | F1 macro ≈ 0,98 |
| Radiação solar em Petrolina (Open-Meteo) | Random Forest | R² ≈ 0,82, MAE ≈ 72 W/m² |

Nas duas tarefas o Random Forest venceu porque os padrões são não lineares (regiões geográficas irregulares; curva diária da radiação), enquanto os modelos lineares serviram como *baseline* e mostraram esse limite. As tabelas e figuras também são salvas na pasta `figuras/`.

# Entrega pelo GitHub

Envie **apenas o link do repositório público**. Organize-o com:

- `README.md` contendo objetivo, fontes e período dos dados, instruções para executar o notebook e resumo dos resultados;
- notebook `.ipynb` com a consulta às APIs, sua análise, os **três classificadores e os três regressores**, todos os imports necessários adicionados por você e conclusões em Markdown;
- os dois CSVs gerados (ou instruções completas e testadas para reproduzi-los) e eventuais figuras utilizadas.

Verifique se o notebook roda **na ordem das células** a partir de um ambiente limpo. Não inclua senhas, tokens ou arquivos sem relação com a avaliação. A banca deve conseguir encontrar as seis comparações e as respostas às duas tarefas sem procurar uma agulha no palheiro.

### Fontes

- [ANEEL — SIGA](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel)
- [ANEEL — recurso e campos usados](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel/resource/11ec447d-698d-4ab8-977f-b424d5deee6a)
- [Open-Meteo — histórico e unidades](https://open-meteo.com/en/docs/historical-weather-api)